# Tinjau kesalahan label validasi
Jalankan dari root Section-Annotations dengan kernel .venv-songformer311.
Notebook ini membaca cache dan audio lokal; tidak mengubah anotasi atau melatih model.
Confidence adalah probabilitas model, bukan jaminan kebenaran. Data test tidak dipakai.


In [ ]:
import json, sys, subprocess
from pathlib import Path
from datetime import datetime
from IPython.display import display, Audio, HTML
assert Path('finetuning/config.json').is_file(), 'Buka notebook dari root proyek'
print(sys.executable)


Jalankan evaluasi sekali untuk membuat laporan baru. Run sebelumnya tidak ditimpa. Output tampil langsung.

In [ ]:
report_dir = Path('outputs.local') / ('validation-review-' + datetime.now().strftime('%Y%m%d-%H%M%S'))
report_dir.mkdir(exist_ok=False)
command = [sys.executable, '-u', '-m', 'finetuning.evaluate',
    '--split', 'val', '--device', 'mps',
    '--checkpoint', 'outputs.local/finetuning/runs/worship-v3/best.pt',
    '--output', str(report_dir / 'metrics.json'),
    '--sections-output', str(report_dir / 'sections.json')]
with (report_dir / 'evaluation.log').open('w') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        result = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
if result:
    raise RuntimeError(f'Evaluasi gagal: {report_dir / "evaluation.log"}')
print('Laporan:', report_dir)


Jika membuka notebook lagi, isi REPORT_PATH dengan lokasi sections.json dari run sebelumnya; evaluasi tidak perlu diulang.

In [ ]:
REPORT_PATH = report_dir / 'sections.json'  # Bisa diganti Path('outputs.local/validation-review-.../sections.json')
payload = json.loads(Path(REPORT_PATH).read_text())
assert payload['split'] == 'val', 'Gunakan validation untuk diagnosis'
rows = payload['sections']
errors = [r for r in rows if not r['correct']]
print(f'{len(errors)} salah dari {len(rows)} section layak')


Filter awal: bridge dan chorus yang tertukar. Ubah FOCUS menjadi None untuk semua kesalahan. Indeks di bawah adalah indeks daftar review, bukan nomor section web.

In [ ]:
from html import escape
FOCUS = {'bridge', 'chorus'}  # None = semua kesalahan
selected = sorted(
    [r for r in errors if FOCUS is None or {r['label_manual'], r['prediction']} == FOCUS],
    key=lambda r: r['confidence'], reverse=True)
columns = ['track_id', 'start', 'end', 'label_manual', 'prediction', 'confidence', 'coverage']
header = '<tr><th>review_index</th>' + ''.join(f'<th>{c}</th>' for c in columns) + '</tr>'
body = ''.join('<tr><td>' + str(i) + '</td>' + ''.join(
    '<td>' + escape(f'{r[c]:.3f}' if isinstance(r[c], float) else str(r[c])) + '</td>'
    for c in columns) + '</tr>' for i, r in enumerate(selected))
display(HTML('<table>' + header + body + '</table>'))
print('Jumlah dipilih:', len(selected))


Pilih review_index. Pemutar memuat potongan maksimal 60 detik dengan konteks 3 detik. Ubah CLIP_OFFSET untuk mendengar bagian selanjutnya pada section panjang. Label manual dan prediksi sama-sama perlu diverifikasi dengan mendengar konteks lagu.

In [ ]:
import librosa
REVIEW_INDEX = 0
CLIP_OFFSET = 0  # detik setelah awal section, untuk section panjang
if not selected:
    print('Tidak ada section untuk filter ini.')
else:
    row = selected[REVIEW_INDEX]
    print(json.dumps(row, indent=2, ensure_ascii=False))
    audio_path = Path(row['audio_path'])
    assert audio_path.is_file(), f'Audio tidak ditemukan: {audio_path}'
    start = max(0, row['start'] + CLIP_OFFSET - 3)
    duration = min(60, row['end'] + 3 - start)
    assert duration > 0, 'CLIP_OFFSET melewati akhir section'
    wav, sr = librosa.load(str(audio_path), sr=24000, mono=True, offset=start, duration=duration)
    print(f'Pemutaran mulai {start:.2f}s, panjang {len(wav)/sr:.2f}s')
    display(Audio(wav, rate=sr))


Catatan pemeriksaan disimpan terpisah secara append-only. Tidak otomatis mengubah ground truth atau target training. Gunakan track_id/start/end untuk mencocokkan anotasi asli: target_section_index hanya nomor section yang lolos audit. Hindari mengubah label agar sekadar cocok dengan prediksi.

In [ ]:
# Isi setelah mendengarkan, lalu jalankan cell untuk menyimpan.
DECISION = 'pending'  # pending, model_error, label_question, boundary_question
NOTE = ''
if selected:
    assert DECISION in {'pending', 'model_error', 'label_question', 'boundary_question'}
    row = selected[REVIEW_INDEX]
    entry = {k: row[k] for k in ['track_id', 'start', 'end', 'label_manual', 'prediction']}
    entry.update(decision=DECISION, note=NOTE, report=str(Path(REPORT_PATH).resolve()),
                 recorded_at=datetime.now().astimezone().isoformat())
    notes_path = Path(REPORT_PATH).parent / 'review_notes.jsonl'
    with notes_path.open('a') as handle:
        handle.write(json.dumps(entry, ensure_ascii=False) + '\n')
    print('Disimpan:', notes_path)
